In [1]:
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")

print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: http://127.0.0.1:5000


In [3]:
np.random.seed(42)

n_samples = 10000

temperature = np.random.normal(75,15,n_samples)
vibration = np.random.normal(0.5,0.2,n_samples)
pressure = np.random.normal(100,20,n_samples)
rpm = np.random.normal(1500,200,n_samples)
age_days = np.random.randint(0,365,n_samples)

failure_score = (
    (temperature > 90)*0.3 +
    (vibration > 0.8)*0.3 +
    (pressure > 130)*0.2 +
    (age_days > 300)*0.2
)

failure_prob = failure_score + np.random.normal(0,0.1,n_samples)

failure = (failure_prob > 0.35).astype(int)

data = pd.DataFrame({
    'temperature': temperature,
    'vibration': vibration,
    'pressure': pressure,
    'rpm': rpm,
    'age_days': age_days,
    'failure': failure
})

print(data.shape)

(10000, 6)


In [4]:
X = data.drop("failure", axis=1)
y = data["failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("PredictiveMaintenance_Lab14")

2026/06/07 12:52:11 INFO mlflow.tracking.fluent: Experiment with name 'PredictiveMaintenance_Lab14' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1780818731349, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1780818731349, lifecycle_stage='active', name='PredictiveMaintenance_Lab14', tags={}, trace_location=None, workspace='default'>

In [8]:
with mlflow.start_run(run_name="Logistic_Regression"):

    lr_model = LogisticRegression(max_iter=1000)

    lr_model.fit(X_train_scaled, y_train)

    y_prob = lr_model.predict_proba(X_test_scaled)[:,1]

    lr_auc = roc_auc_score(y_test, y_prob)

    mlflow.log_metric("roc_auc", lr_auc)

    mlflow.sklearn.log_model(
        lr_model,
        name="model"
    )

    print("LR AUC:", lr_auc)

2026/06/07 12:52:58 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/06/07 12:54:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle form

LR AUC: 0.8460205629529749
🏃 View run Logistic_Regression at: http://127.0.0.1:5000/#/experiments/2/runs/31f90620a9f845bb8a62ce95fb7ab14d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [9]:
with mlflow.start_run(run_name="Random_Forest"):

    rf_model = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )

    rf_model.fit(X_train_scaled, y_train)

    y_prob = rf_model.predict_proba(X_test_scaled)[:,1]

    rf_auc = roc_auc_score(y_test, y_prob)

    mlflow.log_metric("roc_auc", rf_auc)

    mlflow.sklearn.log_model(
        rf_model,
        name="model"
    )

    print("RF AUC:", rf_auc)

2026/06/07 12:56:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RF AUC: 0.9427439743805832
🏃 View run Random_Forest at: http://127.0.0.1:5000/#/experiments/2/runs/097a2265b22c472d936911f6946c7806
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [10]:
with mlflow.start_run(run_name="XGBoost"):

    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss"
    )

    xgb_model.fit(X_train_scaled, y_train)

    y_prob = xgb_model.predict_proba(X_test_scaled)[:,1]

    xgb_auc = roc_auc_score(y_test, y_prob)

    mlflow.log_metric("roc_auc", xgb_auc)

    mlflow.sklearn.log_model(
        xgb_model,
        name="model"
    )

    print("XGB AUC:", xgb_auc)

2026/06/07 12:57:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGB AUC: 0.9413405247485813
🏃 View run XGBoost at: http://127.0.0.1:5000/#/experiments/2/runs/cb3f230ce366430fa69cbe8388a119a4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [11]:
results = pd.DataFrame({
    "Model":[
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "ROC_AUC":[
        lr_auc,
        rf_auc,
        xgb_auc
    ]
})

results.sort_values(
    "ROC_AUC",
    ascending=False
)

,Model,ROC_AUC
1,Random Forest,0.942744
2,XGBoost,0.941341
0,Logistic Regression,0.846021


In [12]:
best_model = xgb_model
print("Best Model = XGBoost")

Best Model = XGBoost


In [13]:
client = MlflowClient()

model_name = "PredictiveMaintenance"

In [14]:
with mlflow.start_run(run_name="Register_Model"):

    model_info = mlflow.sklearn.log_model(
        best_model,
        name="best_model",
        registered_model_name=model_name
    )

print("Model Registered")

2026/06/07 12:58:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'PredictiveMaintenance'.
2026/06/07 12:58:42 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PredictiveMaintenance, version 1
Created version '1' of model 'PredictiveMaintenance'.


🏃 View run Register_Model at: http://127.0.0.1:5000/#/experiments/2/runs/dca0c26879204c03b631244ca24cfe61
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Model Registered


In [15]:
client.update_registered_model(
    name=model_name,
    description="Predictive Maintenance using XGBoost"
)

print("Description Added")

Description Added


In [16]:
client.set_registered_model_tag(
    model_name,
    "framework",
    "xgboost"
)

client.set_registered_model_tag(
    model_name,
    "validation_status",
    "passed"
)

print("Tags Added")

Tags Added


In [17]:
client.set_registered_model_alias(
    model_name,
    "staging",
    1
)

print("Version 1 moved to Staging")

Version 1 moved to Staging


In [18]:
test_machine = pd.DataFrame({
    "temperature":[95],
    "vibration":[0.9],
    "pressure":[135],
    "rpm":[1500],
    "age_days":[320]
})

test_scaled = scaler.transform(test_machine)

prediction = best_model.predict(test_scaled)

print("Prediction:", prediction[0])

Prediction: 1


In [19]:
client.set_registered_model_alias(
    model_name,
    "production",
    1
)

print("Version 1 moved to Production")

Version 1 moved to Production


In [20]:
def predict_equipment_failure(
    temperature,
    vibration,
    pressure,
    rpm,
    age_days
):

    data = pd.DataFrame([{
        "temperature":temperature,
        "vibration":vibration,
        "pressure":pressure,
        "rpm":rpm,
        "age_days":age_days
    }])

    data_scaled = scaler.transform(data)

    pred = best_model.predict(data_scaled)[0]

    if pred == 1:
        return "Schedule Maintenance"

    return "Normal Operation"

In [22]:
print(
    predict_equipment_failure(
        70,0.4,95,1500,100
    )
)

print(
    predict_equipment_failure(
        95,0.9,135,1500,320
    )
)

print(
    predict_equipment_failure(
        85,0.6,110,1500,200
    )
)

Normal Operation
Schedule Maintenance
Normal Operation


In [23]:
with mlflow.start_run(run_name="Register_Version2"):

    mlflow.sklearn.log_model(
        rf_model,
        name="rf_model",
        registered_model_name=model_name
    )

print("Version 2 Registered")

2026/06/07 12:59:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'PredictiveMaintenance' already exists. Creating a new version of this model...
2026/06/07 12:59:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: PredictiveMaintenance, version 2
Created version '2' of model 'PredictiveMaintenance'.


🏃 View run Register_Version2 at: http://127.0.0.1:5000/#/experiments/2/runs/8fa8fdc4f98342f3ab31b7c67b216839
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
Version 2 Registered


In [24]:
client.set_registered_model_alias(
    model_name,
    "production",
    1
)

print("Rollback Complete")

Rollback Complete
